# Совместная работа с Git

Представим себе такой проект:

```
my-project/
  some_script.py
  other_script.py
  README.md
```

Разработчику Леше нужно `some_script.py`, для этого он создает `some_script_v2.py` и помечает где-то (например в `README.md`), что теперь нужно использовать его.

Со временем таких файлов накопится огромное количество, и нужно будет например вынести в отдельное место старые версии, либо же не хранить их вовсе, тогда мы потеряем историю изменений.

Зачем нам история изменений? Что, если Леша допустил ошибку в новой версии, и нужно срочно вернуться к предыдущей?

В нашем "наивном" подходе возникает еще проблемы - что, если изменения касаются одновременно нескольких файлов? Как объяснить причину изменения, время, авторство?

Наконец, что если Петя, второй разработчик в команде, хочет одну часть кода, а Леша другую, одновременно? И самое страшное - что, если изменения в одном и том же месте в одно и то же время?

Для решения этих проблем существуют *системы контроля версий (version control systems, VCS)*, самой популярной в наше время является `git`.

## git

`git` позволяет:

* сохранить осмысленное состояние проекта;
* увидеть, что, когда и кем было изменено;
* вернуться к прежнему состоянию;
* параллельно разрабатывать разные возможности;
* обмениваться изменениями с другими разработчиками;
* обсуждать и проверять код до его попадания в основную версию.

`git` - локальная программа. Для хранения резервных копий, синхронизации нужен отдельный сервер, или сервис вроде [GitHub](https://github.com/), [GitLab](https://gitlab.com/), [BitBucket](https://bitbucket.org).

Проверим, есть ли `git` на компьютере:

In [1]:
!git --version

git version 2.43.0


Для определения авторства изменения имеет смысл настроить имя и email:

In [2]:
!git config --global user.name "Игорь Шинкарев"
!git config --global user.email test@example.com

Флаг `--global` означает глобальные настройки на компьютере, `--local` - в конкретном проекте.

## Репозиторий

*Репозиторий* - проект, который управляется при помощи `git`.

Создадим новый проект:

In [3]:
!mkdir git-demo
%cd git-demo


/content/git-demo


(`!` и `%` игнорируем, они имеют отношение лишь к Jupyter Notebooks)

`mkdir` создает новую папку, `cd` переходит в нее.

В папке проекта выполняется команда

In [4]:
!git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/git-demo/.git/


(`%cd git-demo` - макрос Jupyter Notebook, к нашей теме он не имеет значения)

Заглянем в наш проект:

In [5]:
!ls -lah .

total 12K
drwxr-xr-x 3 root root 4.0K Sep 30 16:58 .
drwxr-xr-x 1 root root 4.0K Sep 30 16:57 ..
drwxr-xr-x 7 root root 4.0K Sep 30 16:58 .git


После `git init` в этой папке появится папка `.git`. В ней находятся история, настройки репозитория, ссылки на ветки и другие служебные данные.

Не нужно вручную редактировать `.git`. Если удалить эту папку, обычные файлы останутся, но локальная Git-история исчезнет.

Изменения хранятся в виде цепочки *коммитов*. Для первого знакомства удобно думать, что коммит — это снимок файлов проекта в определённый момент.

`git` не создаёт полную видимую копию папки на каждый коммит. Он эффективно хранит объекты и переиспользует неизменившееся содержимое. На базовом уровне достаточно знать три типа объектов:

* *blob* — содержимое файла;
* *tree* — имена файлов и структура каталогов;
* *commit* — ссылка на снимок, автора, сообщение и предыдущий коммит.


`git` отслеживает состояния файлов.

Посмотрим текущее состояние проекта:

In [6]:
!git status

On branch master

No commits yet

nothing to commit (create/copy files and use "git add" to track)


Создадим файл:

In [7]:
!touch main.py

(В Windows это будет что-то вроде `type nul > main.py`)

И проверим опять состояние:

In [8]:
!git status

On branch master

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	main.py

nothing added to commit but untracked files present (use "git add" to track)


Мы видим *untracked files* - файлы, которые были изменены, но мы еще не готовы зафиксировать это изменение. Зафиксируем их для этого файла:

In [9]:
!git add main.py
!git status

On branch master

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   main.py



*Changes to be committed* - файлы, которые будут сохранены в изменении (необязательно это должны быть все модифицированные файлы).

Наконец зафиксируем изменение:

In [10]:
!git commit -m "Создан файл main.py"

[master (root-commit) a6e95aa] Создан файл main.py
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 main.py


Флаг `-m` передает сообщение к изменению.

Изменения зафиксированы. Мы можем посмотреть их историю:

In [11]:
!git log --oneline

a6e95aa (HEAD -> master) Создан файл main.py


Флаг `--oneline` отобразит коммит в простом однострочном виде:

* первая часть - хэш коммита - его идентификационный номер
* затем - информация о ветках
* в конце - сообщение к коммиту

HEAD означает текущий коммит. Посмотрим его структуру подробнее:

In [12]:
!git show HEAD

commit a6e95aae16f8b3a21c8407f80e352c0d7fc528f9 (HEAD -> master)
Author: Игорь Шинкарев <test@example.com>
Date:   Wed Sep 30 17:04:59 2026 +0000

    Создан файл main.py

diff --git a/main.py b/main.py
new file mode 100644
index 0000000..e69de29


## .gitignore

Мы можем указать файлы или упрощенные пути к многим файлам в специальном файле `.gitignore`.

Файлы оттуда будут игнорироваться в `git`, и не попадут в него.

Например, есть такая полезная практика в разработке - хранить чувствительную (sensitive) информацию (логины, пароли) в файле `.env`.

Создадим такой файл сразу с паролем:

In [13]:
%%writefile .env
SOME_PASSWORD=12345

Writing .env


(`%%writefile` - специальный макрос Jupyter Notebook. Просто
измените файл текстовым редактором и сохраните)

И посмотрим на `git status`:


In [14]:
!git status

On branch master
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.env

nothing added to commit but untracked files present (use "git add" to track)


Нам не нужен этот пароль в истории `git`! Для этого заведем `.gitignore`:

In [15]:
%%writefile .gitignore
.env

Writing .gitignore


Посмотрим на изменения:

In [18]:
!git status

On branch master
nothing to commit, working tree clean


Теперь нам осталось лишь сохранить сам `gitignore`:

In [17]:
!git add .gitignore
!git commit -m "Создан .gitignore"

[master 48de5c0] Создан .gitignore
 1 file changed, 1 insertion(+)
 create mode 100644 .gitignore


Важно: `.gitignore` не перестаёт отслеживать файл, который уже попал в репозиторий. И главное — секрет, однажды закоммиченный и отправленный на сервер, нельзя считать секретом даже после удаления из последнего коммита. Такой ключ или пароль нужно заменить.


# Три состояния

Итак, файлы у нас могут существовать в нескольких состояниях:

```text
Рабочая директория       Staging area           Репозиторий
(редактируем файлы)      (готовим снимок)        (история коммитов)

      git add  ───────────────►
                              git commit ───────────────►

      ◄──── git restore
      ◄──── git restore --staged ──────
```

Изменим наш `main.py`:

In [19]:
%%writefile main.py
print("Hello, World")

Overwriting main.py


Посмотрим на состояние:

In [20]:
!git status

On branch master
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   main.py

no changes added to commit (use "git add" and/or "git commit -a")


И видим `modified` у `main.py`.

Посмотрим на изменения в файле:

In [21]:
!git diff

diff --git a/main.py b/main.py
index e69de29..e511247 100644
--- a/main.py
+++ b/main.py
@@ -0,0 +1 @@
+print("Hello, World")


С `-` будут начинаться удаленные строчки, с `+` добавленные.

Мы можем откатить наши изменения:

In [22]:
!git restore main.py

Проверим:

In [23]:
!git status

On branch master
nothing to commit, working tree clean


Что, если изменения уже подготовлены к коммиту?

In [24]:
%%writefile main.py
print("Hello, World")

Overwriting main.py


In [25]:
!git add main.py
!git status

On branch master
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   main.py



Обычный `restore` не поможет:

In [26]:
!git restore main.py
!git status

On branch master
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   main.py



С флагом `--staged`

In [27]:
!git restore --staged main.py
!git status

On branch master
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   main.py

no changes added to commit (use "git add" and/or "git commit -a")


Мы убрали файл из будущего коммита. Теперь можем откатить и само изменение:

In [28]:
!git restore main.py
!git status

On branch master
nothing to commit, working tree clean


Сохраним наше изменение окончательно:

In [29]:
%%writefile main.py
print("Hello, World")

Overwriting main.py


In [30]:
!git add main.py
!git commit -m "Написан Hello World"

[master b666f7d] Написан Hello World
 1 file changed, 1 insertion(+)


И проверим историю изменений:

In [31]:
!git log --oneline

b666f7d (HEAD -> master) Написан Hello World
48de5c0 Создан .gitignore
a6e95aa Создан файл main.py


Таким образом:

* `git restore --staged main.py` убирает файл из staging area, но оставляет правки;
* `git restore main.py` удаляет незакоммиченные правки в рабочей директории.

## Ветки

Пока что наша разработка велась линейно. Но основной рабочий процесс ведется в *ветках (branches)*.

In [32]:
!git branch

* master


Основная ветка для разработки - обычно ее называют master, main, develop, trunk, или как еще придумает команда.

От нее создается ветка для работы над новым кодом:

In [33]:
!git checkout -b feature/personal_greeting

Switched to a new branch 'feature/personal_greeting'


`-b` - переключись на ветку, и далее имя ветки. Если ветки не существует - создается новая.

В современном `git` лучше использовать:

In [34]:
!git switch -c feature/personal_greeting

fatal: a branch named 'feature/personal_greeting' already exists


Посмотрим еще раз на ветки:

In [35]:
!git branch

* feature/personal_greeting
  master


И статус:

In [36]:
!git status

On branch feature/personal_greeting
nothing to commit, working tree clean


Внесем изменения в наш скрипт:

In [37]:
%%writefile main.py
name = input("Insert your name: ")
print(f"Hello, {name}!")

Overwriting main.py


Зафиксируем:

In [38]:
!git add main.py
!git commit -m "Добавить персональное приветствие"


[feature/personal_greeting c0b0a9d] Добавить персональное приветствие
 1 file changed, 2 insertions(+), 1 deletion(-)


Посмотрим на историю коммитов как на граф:

In [39]:
!git log --oneline --graph --decorate --all


* c0b0a9d (HEAD -> feature/personal_greeting) Добавить персональное приветствие
* b666f7d (master) Написан Hello World
* 48de5c0 Создан .gitignore
* a6e95aa Создан файл main.py


И сравним изменения с нашей основной веткой:

In [40]:
!git diff master

diff --git a/main.py b/main.py
index e511247..e20a37e 100644
--- a/main.py
+++ b/main.py
@@ -1 +1,2 @@
-print("Hello, World")
+name = input("Insert your name: ")
+print(f"Hello, {name}!")


Мы проверили ветку, удостоверились, что код работает. Настало время добавить изменения в основной код.

Перейдем в ветку, **куда** должны попасть изменения:

In [41]:
!git switch master

Switched to branch 'master'


 И укажем, откуда *влить (merge)* изменения:

In [42]:
!git merge feature/personal_greeting

Updating b666f7d..c0b0a9d
Fast-forward
 main.py | 3 ++-
 1 file changed, 2 insertions(+), 1 deletion(-)


В простом случае Git выполнит fast-forward: передвинет `master` вперёд без отдельного merge-коммита.

Удалить уже слитую локальную ветку необязательно, но можно:


In [43]:
!git branch -d feature/personal_greeting

Deleted branch feature/personal_greeting (was c0b0a9d).


Проверим:

In [44]:
!git branch
!git status

* master
On branch master
nothing to commit, working tree clean


In [46]:
!git log --oneline

c0b0a9d (HEAD -> master) Добавить персональное приветствие
b666f7d Написан Hello World
48de5c0 Создан .gitignore
a6e95aa Создан файл main.py


## Конфликты

`git` хорошо объединяет независимые изменения. Например, один человек создал `README.md`, а другой добавил тесты. Но если две ветки по-разному изменили одну и ту же строку, `git` не знает, какой вариант правильный.

Это не поломка `git`. Это вопрос к разработчику:

> Как должен выглядеть итоговый код с учётом обоих изменений?

Мы можем вести несколько веток одновременно:

In [47]:
!git switch -c feature/greeting_function

Switched to a new branch 'feature/greeting_function'


In [48]:
%%writefile main.py
def greet(name):
  print(f"Hello, {name}!")

name = input("Insert your name: ")
greet(name)

Overwriting main.py


(тут вы можете увидеть пример простейшей функции с одним параметром)

In [49]:
!git add main.py
!git commit -m "Добавлена функция приветствования"

[feature/greeting_function 742ce12] Добавлена функция приветствования
 1 file changed, 4 insertions(+), 1 deletion(-)


In [50]:
!git log --oneline

742ce12 (HEAD -> feature/greeting_function) Добавлена функция приветствования
c0b0a9d (master) Добавить персональное приветствие
b666f7d Написан Hello World
48de5c0 Создан .gitignore
a6e95aa Создан файл main.py


In [52]:
!git switch master
!git switch -c feature/russian_greet

Switched to branch 'master'
Switched to a new branch 'feature/russian_greet'


In [53]:
!git log --oneline

c0b0a9d (HEAD -> feature/russian_greet, master) Добавить персональное приветствие
b666f7d Написан Hello World
48de5c0 Создан .gitignore
a6e95aa Создан файл main.py


In [55]:
%%writefile main.py
name = input("Введите ваше имя: ")
print(f"Привет, {name}!")

Overwriting main.py


In [56]:
!git add main.py
!git commit -m "Приветствование переведено на русский"

[feature/russian_greet 18d4860] Приветствование переведено на русский
 1 file changed, 2 insertions(+), 2 deletions(-)


Посмотрим на граф коммитов:

In [57]:
!git log --oneline --graph --decorate --all


* 18d4860 (HEAD -> feature/russian_greet) Приветствование переведено на русский
| * 742ce12 (feature/greeting_function) Добавлена функция приветствования
|/  
* c0b0a9d (master) Добавить персональное приветствие
* b666f7d Написан Hello World
* 48de5c0 Создан .gitignore
* a6e95aa Создан файл main.py


In [58]:
!git branch

  feature/greeting_function
* feature/russian_greet
  master


От ветки `master` идут две ветки с разным кодом!

Сначала зальем ветку с русским переводом:

In [59]:
!git switch master
!git merge feature/russian_greet

Switched to branch 'master'
Updating c0b0a9d..18d4860
Fast-forward
 main.py | 4 ++--
 1 file changed, 2 insertions(+), 2 deletions(-)


Затем вынесение приветствия в функцию:

In [60]:
!git switch master
!git merge feature/greeting_function

Already on 'master'
Auto-merging main.py
CONFLICT (content): Merge conflict in main.py
Automatic merge failed; fix conflicts and then commit the result.


Не паникуем!

Посмотрим статус:

In [61]:
!git status

On branch master
You have unmerged paths.
  (fix conflicts and run "git commit")
  (use "git merge --abort" to abort the merge)

Unmerged paths:
  (use "git add <file>..." to mark resolution)
	both modified:   main.py

no changes added to commit (use "git add" and/or "git commit -a")


Откатим эту попытку:

In [62]:
!git merge --abort
!git status

On branch master
nothing to commit, working tree clean


И попробуем еще раз:

In [63]:
!git merge feature/greeting_function

Auto-merging main.py
CONFLICT (content): Merge conflict in main.py
Automatic merge failed; fix conflicts and then commit the result.


Посмотрим наш конфликтный файл:

In [64]:
%pycat main.py

* между `<<<<<<< HEAD` и `=======` находится вариант текущей ветки;
* между `=======` и `>>>>>>> ...` находится входящий вариант;
* маркеры не являются синтаксисом Python и должны быть удалены;
* разрешить конфликт — не обязательно выбрать одну сторону целиком; можно написать третий, объединённый вариант.

Так и поступим:

In [65]:
%%writefile main.py
def greet(name):
  print(f"Привет, {name}!")

name = input("Введите ваше имя: ")
greet(name)

Overwriting main.py


Добавим в stage:

In [66]:
!git add main.py
!git status

On branch master
All conflicts fixed but you are still merging.
  (use "git commit" to conclude merge)

Changes to be committed:
	modified:   main.py



И сохраним:

In [67]:
!git commit -m "Исправлен конфликт"

[master 92caa79] Исправлен конфликт


Вот как выглядит наша история теперь:

In [68]:
!git log --oneline --graph --decorate --all

*   92caa79 (HEAD -> master) Исправлен конфликт
|\  
| * 742ce12 (feature/greeting_function) Добавлена функция приветствования
* | 18d4860 (feature/russian_greet) Приветствование переведено на русский
|/  
* c0b0a9d Добавить персональное приветствие
* b666f7d Написан Hello World
* 48de5c0 Создан .gitignore
* a6e95aa Создан файл main.py


## Коллективная работа

Мы можем обмениваться репозиториями напрямую (главное, чтобы папка `.git` была в проекте), но удобнее и надежнее иметь для этого отдельный сервер.

Посмотрим на примере `GitHub`, как это работает.

Я заранее создал [репозиторий](https://github.com/itacademyfall2026/git-demo) для этого.

Свяжем наш локальный проект с ним:

In [69]:
!git remote add origin https://github.com/itacademyfall2026/git-demo

`origin` (имя может быть любым) отныне будет означать для нас удаленный источник кода, на который мы можем ориентироваться.

In [70]:
!git remote -v

origin	https://github.com/itacademyfall2026/git-demo (fetch)
origin	https://github.com/itacademyfall2026/git-demo (push)


Отправим наши изменения на `GitHub`:

In [ ]:
!git push -u origin master

- **remote** — запись об удалённом репозитории;
- `origin` — обычное имя основного remote, а не специальное слово языка Git;
- `push` отправляет локальные коммиты;
- флаг `-u` связывает локальную ветку с удалённой, чтобы дальше можно было писать просто `git push` и `git pull`.

Если репозиторий уже существует на GitHub, обычно не выполняют `git init`. Его копируют вместе с историей:

In [ ]:
!git clone https://github.com/itacademyfall2026/git-demo
!cd git-demo

`clone` создаёт папку, скачивает историю, настраивает `origin` и рабочую ветку.

In [ ]:
!git fetch origin

`fetch` скачивает информацию о новых коммитах, но не меняет файлы текущей ветки. Это безопасный способ сначала узнать о состоянии сервера.

In [ ]:
!git pull

Для базового сценария `pull` можно понимать как «скачать новые коммиты и включить их в текущую ветку». Концептуально это `fetch`, после которого Git интегрирует полученные изменения.

Перед началом работы полезно выполнить `git pull`, особенно если над проектом работают несколько человек.